In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
print("📚 Loading data from CSV files...")

books = pd.read_csv('data/BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
ratings = pd.read_csv('data/BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
users = pd.read_csv('data/BX-Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

print(f"✅ Books: {books.shape[0]} rows")
print(f"✅ Ratings: {ratings.shape[0]} rows")
print(f"✅ Users: {users.shape[0]} rows")

📚 Loading data from CSV files...
✅ Books: 271360 rows
✅ Ratings: 1149780 rows
✅ Users: 278858 rows


In [3]:
print("🧹 Cleaning books data...")

books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]

books.rename(columns={
    'Book-Title': 'title',
    'Book-Author': 'author',
    'Year-Of-Publication': 'year',
    'Publisher': 'publisher',
    'Image-URL-L': 'image_url'
}, inplace=True)

print("✅ Books columns renamed")
books.head()

🧹 Cleaning books data...
✅ Books columns renamed


,ISBN,title,author,year,publisher,image_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...


In [4]:
print("📝 Cleaning ratings data...")

ratings.rename(columns={
    'User-ID': 'user_id',
    'Book-Rating': 'rating'
}, inplace=True)

print("✅ Ratings columns renamed")
print(f"📋 Ratings columns: {ratings.columns.tolist()}")
ratings.head()

📝 Cleaning ratings data...
✅ Ratings columns renamed
📋 Ratings columns: ['user_id', 'ISBN', 'rating']


,user_id,ISBN,rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [5]:
print("🗑️ Removing 0 ratings...")

before = len(ratings)
ratings = ratings[ratings['rating'] > 0]
after = len(ratings)

print(f"✅ Removed {before - after} rows with 0 rating")
print(f"✅ Remaining ratings: {after}")

🗑️ Removing 0 ratings...
✅ Removed 716109 rows with 0 rating
✅ Remaining ratings: 433671


In [6]:
print("👥 Filtering active users (at least 5 ratings)...")

user_counts = ratings['user_id'].value_counts()
active_users = user_counts[user_counts >= 5].index
ratings = ratings[ratings['user_id'].isin(active_users)]

print(f"✅ Active users: {ratings['user_id'].nunique()}")

👥 Filtering active users (at least 5 ratings)...
✅ Active users: 14220


In [7]:
print("📖 Filtering popular books (at least 10 ratings)...")

book_counts = ratings['ISBN'].value_counts()
popular_books = book_counts[book_counts >= 10].index
ratings = ratings[ratings['ISBN'].isin(popular_books)]

print(f"✅ Popular books: {ratings['ISBN'].nunique()}")

📖 Filtering popular books (at least 10 ratings)...
✅ Popular books: 4136


In [8]:
print("🔗 Merging ratings with book information...")

ratings_with_books = ratings.merge(books, on='ISBN')
print(f"✅ Merged data shape: {ratings_with_books.shape}")
ratings_with_books.head()

🔗 Merging ratings with book information...
✅ Merged data shape: (94257, 8)


,user_id,ISBN,rating,title,author,year,publisher,image_url
0,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells,2003,HarperTorch,http://images.amazon.com/images/P/0060517794.0...
1,276747,0671537458,9,Waiting to Exhale,Terry McMillan,1995,Pocket,http://images.amazon.com/images/P/0671537458.0...
2,276747,0679776818,8,Birdsong: A Novel of Love and War,Sebastian Faulks,1997,Vintage Books USA,http://images.amazon.com/images/P/0679776818.0...
3,276822,0060096195,10,The Boy Next Door,Meggin Cabot,2002,Avon Trade,http://images.amazon.com/images/P/0060096195.0...
4,276822,0786817070,10,"Artemis Fowl (Artemis Fowl, Book 1)",Eoin Colfer,2002,Miramax Kids,http://images.amazon.com/images/P/0786817070.0...


In [9]:
print("📊 Calculating rating counts per book...")

rating_count = ratings_with_books.groupby('title')['rating'].count().reset_index()
rating_count.rename(columns={'rating': 'num_of_rating'}, inplace=True)

final_rating = ratings_with_books.merge(rating_count, on='title')
print(f"✅ Shape: {final_rating.shape}")
final_rating.head()

📊 Calculating rating counts per book...
✅ Shape: (94257, 9)


,user_id,ISBN,rating,title,author,year,publisher,image_url,num_of_rating
0,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells,2003,HarperTorch,http://images.amazon.com/images/P/0060517794.0...,21
1,276747,0671537458,9,Waiting to Exhale,Terry McMillan,1995,Pocket,http://images.amazon.com/images/P/0671537458.0...,14
2,276747,0679776818,8,Birdsong: A Novel of Love and War,Sebastian Faulks,1997,Vintage Books USA,http://images.amazon.com/images/P/0679776818.0...,16
3,276822,0060096195,10,The Boy Next Door,Meggin Cabot,2002,Avon Trade,http://images.amazon.com/images/P/0060096195.0...,38
4,276822,0786817070,10,"Artemis Fowl (Artemis Fowl, Book 1)",Eoin Colfer,2002,Miramax Kids,http://images.amazon.com/images/P/0786817070.0...,87


In [10]:
print("⭐ Keeping books with at least 50 ratings...")

final_rating = final_rating[final_rating['num_of_rating'] >= 50]
print(f"✅ Books with ≥50 ratings: {final_rating['title'].nunique()}")

⭐ Keeping books with at least 50 ratings...
✅ Books with ≥50 ratings: 394


In [11]:
print("🔄 Removing duplicate user-book ratings...")

before = len(final_rating)
final_rating.drop_duplicates(['user_id', 'title'], inplace=True)
after = len(final_rating)

print(f"✅ Removed {before - after} duplicates")
print(f"✅ Final shape: {final_rating.shape}")

🔄 Removing duplicate user-book ratings...
✅ Removed 214 duplicates
✅ Final shape: (35098, 9)


In [12]:
print("📊 Calculating average rating per book...")

avg_rating = final_rating.groupby('title')['rating'].mean().reset_index()
avg_rating.rename(columns={'rating': 'avg_rating'}, inplace=True)

# Round to 1 decimal
avg_rating['avg_rating'] = avg_rating['avg_rating'].round(1)

print("✅ Average rating calculated")
avg_rating.head()

📊 Calculating average rating per book...
✅ Average rating calculated


,title,avg_rating
0,1984,8.9
1,1st to Die: A Novel,7.7
2,2nd Chance,7.8
3,A Bend in the Road,7.4
4,A Case of Need,7.0


In [13]:
print("📚 Creating enriched book details with publisher, year, and ratings...")

# Get unique book details
book_details = final_rating[['title', 'author', 'publisher', 'year', 'image_url', 'num_of_rating']].drop_duplicates('title')

# Merge with average rating
book_details = book_details.merge(avg_rating, on='title')

# Fill missing values
book_details['publisher'] = book_details['publisher'].fillna('Unknown')
book_details['year'] = book_details['year'].fillna('Unknown')
book_details['author'] = book_details['author'].fillna('Unknown Author')
book_details['image_url'] = book_details['image_url'].fillna('https://via.placeholder.com/500x750?text=No+Cover')

print(f"✅ Book details enriched: {book_details.shape[0]} books")
print(f"📋 Columns: {book_details.columns.tolist()}")
book_details.head()

📚 Creating enriched book details with publisher, year, and ratings...
✅ Book details enriched: 394 books
📋 Columns: ['title', 'author', 'publisher', 'year', 'image_url', 'num_of_rating', 'avg_rating']


,title,author,publisher,year,image_url,num_of_rating,avg_rating
0,"Artemis Fowl (Artemis Fowl, Book 1)",Eoin Colfer,Miramax Kids,2002,http://images.amazon.com/images/P/0786817070.0...,87,7.6
1,Whispers,Dean R. Koontz,Berkley Publishing Group,2001,http://images.amazon.com/images/P/042518109X.0...,58,7.2
2,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,John Wiley &amp; Sons Inc,1994,http://images.amazon.com/images/P/002542730X.0...,61,7.8
3,The Da Vinci Code,Dan Brown,Doubleday,2003,http://images.amazon.com/images/P/0385504209.0...,339,8.5
4,The Chamber,John Grisham,Dell Publishing Company,1995,http://images.amazon.com/images/P/0440220602.0...,153,7.4


In [14]:
print("📐 Creating pivot table (Books × Users)...")

book_pivot = final_rating.pivot_table(
    columns='user_id',
    index='title',
    values='rating'
).fillna(0)

print(f"✅ Pivot table created!")
print(f"   - Books: {book_pivot.shape[0]}")
print(f"   - Users: {book_pivot.shape[1]}")

📐 Creating pivot table (Books × Users)...
✅ Pivot table created!
   - Books: 394
   - Users: 8672


In [15]:
print("💾 Converting to sparse matrix...")

book_sparse = csr_matrix(book_pivot)
print("✅ Sparse matrix created (memory efficient)")

💾 Converting to sparse matrix...
✅ Sparse matrix created (memory efficient)


In [16]:
print("🤖 Training KNN model with Cosine Similarity...")

model = NearestNeighbors(algorithm='brute', metric='cosine')
model.fit(book_sparse)

print("✅ KNN model trained successfully!")
print("   - Will return 10 recommendations per search")

🤖 Training KNN model with Cosine Similarity...
✅ KNN model trained successfully!
   - Will return 10 recommendations per search


In [17]:
def recommend_book(book_name, n_recommendations=10):
    """
    Returns top N book recommendations based on the input book
    """
    if book_name not in book_pivot.index:
        print(f"❌ Book '{book_name}' not found in database")
        return []
    
    book_id = np.where(book_pivot.index == book_name)[0][0]
    
    distances, suggestions = model.kneighbors(
        book_pivot.iloc[book_id, :].values.reshape(1, -1),
        n_neighbors=n_recommendations + 1
    )
    
    recommendations = []
    for i in range(1, len(suggestions[0])):
        recommended_book = book_pivot.index[suggestions[0][i]]
        recommendations.append(recommended_book)
    
    return recommendations

print("✅ Recommendation function created")

✅ Recommendation function created


In [18]:
print("🧪 Testing the recommendation function...")

test_book = "Harry Potter and the Chamber of Secrets (Book 2)"

if test_book in book_pivot.index:
    recs = recommend_book(test_book, 10)
    print(f"\n📖 Book: {test_book}")
    print(f"\n🎯 TOP 10 RECOMMENDATIONS:\n")
    for i, book in enumerate(recs, 1):
        print(f"{i}. {book}")
        
    # Show full details of first recommendation
    if recs:
        first_rec = recs[0]
        rec_details = book_details[book_details['title'] == first_rec]
        if not rec_details.empty:
            print(f"\n📚 Example Recommendation Details:")
            print(f"   Title: {rec_details.iloc[0]['title']}")
            print(f"   Author: {rec_details.iloc[0]['author']}")
            print(f"   Publisher: {rec_details.iloc[0]['publisher']}")
            print(f"   Year: {rec_details.iloc[0]['year']}")
            print(f"   Avg Rating: {rec_details.iloc[0]['avg_rating']}/10")
            print(f"   Total Ratings: {rec_details.iloc[0]['num_of_rating']}")
else:
    print(f"⚠️ Test book not found. Using first book in database...")
    test_book = book_pivot.index[0]
    recs = recommend_book(test_book, 10)
    print(f"\n📖 Book: {test_book}")
    print(f"\n🎯 TOP 10 RECOMMENDATIONS:\n")
    for i, book in enumerate(recs, 1):
        print(f"{i}. {book}")

🧪 Testing the recommendation function...

📖 Book: Harry Potter and the Chamber of Secrets (Book 2)

🎯 TOP 10 RECOMMENDATIONS:

1. Harry Potter and the Prisoner of Azkaban (Book 3)
2. Harry Potter and the Goblet of Fire (Book 4)
3. Harry Potter and the Sorcerer's Stone (Book 1)
4. Harry Potter and the Order of the Phoenix (Book 5)
5. Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))
6. The Fellowship of the Ring (The Lord of the Rings, Part 1)
7. The Bad Beginning (A Series of Unfortunate Events, Book 1)
8. Charlotte's Web (Trophy Newbery)
9. The Hobbit : The Enchanting Prelude to The Lord of the Rings
10. Bridget Jones's Diary

📚 Example Recommendation Details:
   Title: Harry Potter and the Prisoner of Azkaban (Book 3)
   Author: J. K. Rowling
   Publisher: Scholastic
   Year: 2001
   Avg Rating: 9.1/10
   Total Ratings: 232


In [19]:
print("💾 Saving model and data files...")

# Create artifacts folder
os.makedirs('artifacts', exist_ok=True)

# Save all files
pickle.dump(model, open('artifacts/model.pkl', 'wb'))
pickle.dump(book_pivot, open('artifacts/book_pivot.pkl', 'wb'))
pickle.dump(book_pivot.index.tolist(), open('artifacts/book_names.pkl', 'wb'))
pickle.dump(final_rating, open('artifacts/final_rating.pkl', 'wb'))
pickle.dump(book_details, open('artifacts/book_details.pkl', 'wb'))

print("✅ All files saved to 'artifacts' folder!")
print("\n📁 Files created:")
print("   - artifacts/model.pkl")
print("   - artifacts/book_pivot.pkl")
print("   - artifacts/book_names.pkl")
print("   - artifacts/final_rating.pkl")
print("   - artifacts/book_details.pkl")

💾 Saving model and data files...
✅ All files saved to 'artifacts' folder!

📁 Files created:
   - artifacts/model.pkl
   - artifacts/book_pivot.pkl
   - artifacts/book_names.pkl
   - artifacts/final_rating.pkl
   - artifacts/book_details.pkl


In [20]:
print("\n" + "="*60)
print("🎉🎉🎉 TRAINING COMPLETE! 🎉🎉🎉")
print("="*60)
print("\n📊 FINAL STATISTICS:")
print(f"   - Books in model: {book_pivot.shape[0]}")
print(f"   - Users in model: {book_pivot.shape[1]}")
print(f"   - Recommendations per search: 10")
print(f"   - Books with full details: {len(book_details)}")
print("\n📋 Book details include:")
print("   ✅ Title")
print("   ✅ Author")
print("   ✅ Publisher")
print("   ✅ Year of Publication")
print("   ✅ Average Rating")
print("   ✅ Number of Ratings")
print("   ✅ Cover Image URL")
print("\n✅ Your model is ready for Streamlit app!")
print("\n🚀 Next step: Run 'streamlit run app_book.py'")


🎉🎉🎉 TRAINING COMPLETE! 🎉🎉🎉

📊 FINAL STATISTICS:
   - Books in model: 394
   - Users in model: 8672
   - Recommendations per search: 10
   - Books with full details: 394

📋 Book details include:
   ✅ Title
   ✅ Author
   ✅ Publisher
   ✅ Year of Publication
   ✅ Average Rating
   ✅ Number of Ratings
   ✅ Cover Image URL

✅ Your model is ready for Streamlit app!

🚀 Next step: Run 'streamlit run app_book.py'
